# yeast-2_vs_4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('39_yeast-2_vs_4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,83.625,79.892,93.333,70.545,99.353,84.949,87.320,96.995,81.758
smote,87.851,75.837,72.661,80.182,96.540,88.361,87.108,97.452,81.844
bsmote,89.627,75.272,68.512,84.182,95.680,89.931,84.586,97.079,82.450
ros,88.642,77.478,74.483,81.273,96.869,89.071,87.751,97.713,83.060
adasyn,89.062,71.965,63.380,84.182,94.494,89.338,85.560,97.312,80.513
casb,86.239,76.828,77.985,76.364,97.518,86.941,87.572,97.679,81.534
casb_relaxed,85.713,76.926,79.629,75.364,97.733,86.548,87.831,97.596,81.320
casb_edited,87.392,75.202,71.947,79.273,96.545,87.909,86.827,97.423,81.297
casb_ros,85.528,75.730,77.773,75.364,97.300,86.332,87.967,97.699,80.629


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,78.531,73.747,93.056,62.727,99.353,81.040,85.449,93.950,76.139
smote,89.016,74.609,68.668,83.182,95.567,89.374,78.779,93.805,81.812
bsmote,89.995,72.179,62.503,86.182,94.163,90.172,73.085,93.877,81.087
ros,86.964,70.872,64.556,80.182,94.917,87.549,76.309,93.323,78.918
adasyn,88.095,69.234,60.079,83.182,93.620,88.401,76.007,93.987,78.665
casb,87.570,74.587,70.927,80.182,96.216,88.199,80.369,93.581,81.079
casb_relaxed,87.413,73.795,69.852,80.182,95.891,88.036,78.925,93.412,80.604
casb_edited,89.396,74.319,67.548,84.182,95.241,89.711,76.470,93.191,81.857
casb_ros,87.075,74.946,73.192,79.182,96.320,87.751,81.307,93.774,81.011


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,82.072,75.447,85.967,68.636,98.595,83.616,91.334,98.636,78.759
smote,85.633,73.663,72.594,76.273,96.654,86.463,88.919,98.289,79.648
bsmote,83.883,70.294,68.057,73.455,96.110,84.782,86.570,97.965,77.088
ros,84.910,74.183,75.364,74.455,97.085,85.770,87.294,98.280,79.546
adasyn,86.384,72.326,68.041,78.182,95.787,86.984,87.569,97.976,79.355
casb,85.563,75.996,78.275,75.364,97.406,86.385,88.650,98.368,80.780
casb_relaxed,86.243,74.776,73.684,77.182,96.758,86.970,88.823,98.440,80.509
casb_edited,89.087,74.772,68.374,83.182,95.677,89.429,87.856,98.134,81.930
casb_ros,84.748,74.972,79.023,74.273,97.408,85.840,89.452,98.505,79.860


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,78.568,71.662,86.667,62.909,98.705,80.807,84.424,94.850,75.115
smote,85.479,69.872,65.117,77.636,95.023,86.330,83.444,94.975,77.675
bsmote,85.938,66.886,58.733,79.727,93.513,86.620,81.560,93.885,76.412
ros,87.359,70.536,64.215,81.545,94.266,87.906,83.145,94.779,78.948
adasyn,87.136,69.426,62.597,81.636,93.943,87.790,82.415,94.928,78.281
casb,85.596,70.161,65.466,77.364,95.131,86.247,84.324,94.873,77.879
casb_relaxed,85.661,71.208,67.549,77.364,95.674,86.519,84.904,95.175,78.435
casb_edited,85.242,66.778,59.406,78.364,93.945,86.154,83.810,95.221,76.010
casb_ros,86.426,69.952,63.883,79.273,94.590,86.931,84.708,94.929,78.189


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,80.699,75.187,89.756,66.205,99.001,82.603,87.132,96.108,77.943
smote,86.995,73.495,69.760,79.318,95.946,87.632,84.562,96.130,80.245
bsmote,87.361,71.158,64.451,80.886,94.867,87.877,81.450,95.702,79.259
ros,86.969,73.267,69.654,79.364,95.784,87.574,83.625,96.024,80.118
adasyn,87.669,70.738,63.524,81.795,94.461,88.128,82.888,96.051,79.203
casb,86.242,74.393,73.163,77.318,96.568,86.943,85.229,96.125,80.318
casb_relaxed,86.257,74.176,72.679,77.523,96.514,87.018,85.121,96.156,80.217
casb_edited,87.779,72.768,66.819,81.250,95.352,88.301,83.741,95.992,80.273
casb_ros,85.944,73.900,73.467,77.023,96.404,86.714,85.858,96.227,79.922


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,87.392,bsmote,89.627,-2.234,LOSS
1,f1,casb_relaxed,76.926,ros,77.478,-0.551,LOSS
2,precision,casb_relaxed,79.629,ros,74.483,5.146,WIN
3,recall,casb_edited,79.273,bsmote,84.182,-4.909,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,89.396,bsmote,89.995,-0.600,LOSS
1,f1,casb_ros,74.946,smote,74.609,0.337,WIN
2,precision,casb_ros,73.192,smote,68.668,4.523,WIN
3,recall,casb_edited,84.182,bsmote,86.182,-2.000,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,89.087,adasyn,86.384,2.703,WIN
1,f1,casb,75.996,ros,74.183,1.813,WIN
2,precision,casb_ros,79.023,ros,75.364,3.658,WIN
3,recall,casb_edited,83.182,adasyn,78.182,5.000,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,87.779,adasyn,87.669,0.110,WIN
1,f1,casb,74.393,smote,73.495,0.898,WIN
2,precision,casb_ros,73.467,smote,69.760,3.707,WIN
3,recall,casb_edited,81.250,adasyn,81.795,-0.545,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.852803,0.842105,1.000000,0.727273,1.000000,0.863636,0.963326,0.994071,0.847454,None
1,42,1,svm,smote,1.00,0.921847,0.740741,0.625000,0.909091,0.934783,0.921937,0.904754,0.978261,0.831294,None
2,42,1,svm,bsmote,0.75,0.927191,0.769231,0.666667,0.909091,0.945652,0.927372,0.865065,0.972332,0.848211,None
3,42,1,svm,ros,0.50,0.943042,0.869565,0.833333,0.909091,0.978261,0.943676,0.913209,0.981225,0.906304,None
4,42,1,svm,adasyn,0.50,0.932505,0.800000,0.714286,0.909091,0.956522,0.932806,0.899639,0.977273,0.866252,None
5,42,1,svm,casb,0.50,0.899605,0.857143,0.900000,0.818182,0.989130,0.903656,0.930197,0.983202,0.878374,None
6,42,1,svm,casb_relaxed,0.50,0.899605,0.857143,0.900000,0.818182,0.989130,0.903656,0.928773,0.982213,0.878374,None
7,42,1,svm,casb_edited,0.50,0.943042,0.869565,0.833333,0.909091,0.978261,0.943676,0.936541,0.982213,0.906304,None
8,42,1,svm,casb_ros,0.50,0.899605,0.857143,0.900000,0.818182,0.989130,0.903656,0.930197,0.983202,0.878374,None
9,42,1,knn,original,0.50,0.797724,0.777778,1.000000,0.636364,1.000000,0.818182,0.903648,0.947628,0.787751,None
